In [ ]:
import re
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

In [ ]:
WEIGHTS_PATH = "weights/best_YOLO27Jan2024.pt"
INPUT_DIR = "train"

In [ ]:
THREE_DIGIT_PNG = re.compile(r"^\d{3}\.png$", re.IGNORECASE)
TWO_DIGIT_PNG = re.compile(r"^\d{2}\.png$", re.IGNORECASE)
ONE_DIGIT_PNG = re.compile(r"^\d{1}\.png$", re.IGNORECASE)

def find_xxx_pngs(folder):
    folder = Path(folder)
    return sorted(
        p for p in folder.iterdir()
        if p.is_file() and (THREE_DIGIT_PNG.match(p.name) or TWO_DIGIT_PNG.match(p.name) or ONE_DIGIT_PNG.match(p.name))
    )

image_paths = find_xxx_pngs(INPUT_DIR)
print(f"Found {len(image_paths)} matching image(s):")
for p in image_paths:
    print(" ", p.name)


In [ ]:
model = YOLO(WEIGHTS_PATH)

In [ ]:
results = model.predict(
    [str(p) for p in image_paths],
    max_det=3,
    iou=0.1,
    imgsz=480,
    conf=0.13,
    save_txt=True,
    save_conf=True,
    save=True,
)

In [ ]:
def find_output_image(save_dir, stem):
    """Find the saved result image for a given filename stem, regardless
    of extension (YOLO usually keeps the original extension, but this is
    a safety net in case it doesn't, or the file wasn't saved at all)."""
    save_dir = Path(save_dir)
    matches = sorted(save_dir.glob(f"{stem}.*"))
    return matches[0] if matches else None


def plot_result_vs_labeled(idx):
    result = results[idx]
    img_name = image_paths[idx].name
    stem = image_paths[idx].stem  # e.g. "543"

    save_dir = Path(result.save_dir)
    result_img_path = find_output_image(save_dir, stem)
    labeled_img_path = Path(INPUT_DIR) / f"{stem}_labeled.png"

    if result_img_path is None:
        print(f"[{img_name}] Kein Ergebnisbild in {save_dir} gefunden.")
        print("Inhalt des Ordners:")
        for f in sorted(save_dir.iterdir()):
            print(" ", f.name)
        return

    result_img = cv2.imread(str(result_img_path))
    if result_img is None:
        print(f"[{img_name}] Datei gefunden, aber nicht lesbar: {result_img_path}")
        return

    fig, axes = plt.subplots(1, 2, figsize=(12, 8))

    axes[0].imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"YOLO result: {result_img_path.name}")
    axes[0].axis("off")

    if labeled_img_path.exists():
        labeled_img = cv2.imread(str(labeled_img_path))
        axes[1].imshow(cv2.cvtColor(labeled_img, cv2.COLOR_BGR2RGB))
        axes[1].set_title(f"Labeled: {labeled_img_path.name}")
    else:
        axes[1].text(0.5, 0.5, "labeled file not found", ha="center", va="center")
        axes[1].set_title(labeled_img_path.name)
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

plot_result_vs_labeled(0)

In [ ]:
for i in range(len(image_paths)):
    plot_result_vs_labeled(i)


In [ ]:
import numpy as np
import pandas as pd

IOU_MATCH_THRESHOLD = 0.1   # ab welcher IoU eine Vorhersage als Treffer zählt
MIN_MASK_AREA = 20          # kleine Rauschflecken in der Maske ignorieren (Pixel)


def mask_to_boxes(mask_path, min_area=MIN_MASK_AREA):
    """Liest eine binäre Labeled-Maske und liefert eine Liste von
    Bounding-Boxen (x1, y1, x2, y2), eine pro zusammenhängender weißer
    Region."""
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []
    _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)

    boxes = []
    for label_id in range(1, n_labels):  # 0 = Hintergrund
        area = stats[label_id, cv2.CC_STAT_AREA]
        if area < min_area:
            continue
        x = stats[label_id, cv2.CC_STAT_LEFT]
        y = stats[label_id, cv2.CC_STAT_TOP]
        w = stats[label_id, cv2.CC_STAT_WIDTH]
        h = stats[label_id, cv2.CC_STAT_HEIGHT]
        boxes.append((x, y, x + w, y + h))
    return boxes


def compute_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    inter_x1, inter_y1 = max(ax1, bx1), max(ay1, by1)
    inter_x2, inter_y2 = min(ax2, bx2), min(ay2, by2)
    inter_w, inter_h = max(0, inter_x2 - inter_x1), max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter_area
    return inter_area / union if union > 0 else 0.0


def match_detections(gt_boxes, pred_boxes, pred_confs, iou_threshold=IOU_MATCH_THRESHOLD):
    """Greedy-Matching nach höchster IoU. Gibt eine Liste von Records
    zurück: je ein Eintrag pro TP, FP oder FN."""
    records = []
    matched_gt = set()
    matched_pred = set()

    # Alle GT-Pred-Paare nach IoU absteigend sortieren
    pairs = []
    for gi, gt_box in enumerate(gt_boxes):
        for pi, pred_box in enumerate(pred_boxes):
            iou = compute_iou(gt_box, pred_box)
            if iou >= iou_threshold:
                pairs.append((iou, gi, pi))
    pairs.sort(reverse=True, key=lambda t: t[0])

    for iou, gi, pi in pairs:
        if gi in matched_gt or pi in matched_pred:
            continue
        matched_gt.add(gi)
        matched_pred.add(pi)
        records.append({"type": "TP", "confidence": pred_confs[pi], "iou": iou})

    for pi in range(len(pred_boxes)):
        if pi not in matched_pred:
            records.append({"type": "FP", "confidence": pred_confs[pi], "iou": None})

    for gi in range(len(gt_boxes)):
        if gi not in matched_gt:
            records.append({"type": "FN", "confidence": None, "iou": None})

    return records


In [ ]:
all_records = []

for idx, img_path in enumerate(image_paths):
    stem = img_path.stem
    labeled_path = Path(INPUT_DIR) / f"{stem}_labeled.png"

    gt_boxes = mask_to_boxes(labeled_path)

    result = results[idx]
    pred_boxes = [tuple(box.xyxy[0].tolist()) for box in result.boxes]
    pred_confs = [float(box.conf[0]) for box in result.boxes]

    if not gt_boxes and not pred_boxes:
        # Bildebene: kein Plaque vorhanden und keiner vorhergesagt
        all_records.append({
            "image": img_path.name, "type": "TN", "confidence": None, "iou": None
        })
        continue

    recs = match_detections(gt_boxes, pred_boxes, pred_confs)
    for r in recs:
        r["image"] = img_path.name
        all_records.append(r)

df = pd.DataFrame(all_records, columns=["image", "type", "confidence", "iou"])
df

In [ ]:
counts = df["type"].value_counts().reindex(["TP", "FP", "FN", "TN"], fill_value=0)
print("Counts:")
print(counts)

tp, fp, fn = counts["TP"], counts["FP"], counts["FN"]
precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else float("nan")

print(f"\nPrecision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-Score:  {f1:.3f}")

print("\nMittlere Konfidenz pro Kategorie:")
print(df.groupby("type")["confidence"].mean())